1. Implement a simple GRU cell from scratch in Python using only NumPy, focusing on the update and reset gates — given input x_t and previous hidden state h_{t-1}, compute the next hidden state h_t.


In [ ]:
import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

class GRUCell:
    def __init__(self, input_size, hidden_size):
        self.input_size = input_size
        self.hidden_size = hidden_size

        self.Wz = np.random.randn(hidden_size, input_size) * 0.1
        self.Uz = np.random.randn(hidden_size, hidden_size) * 0.1

        self.Wr = np.random.randn(hidden_size, input_size) * 0.1
        self.Ur = np.random.randn(hidden_size, hidden_size) * 0.1

        self.Wh = np.random.randn(hidden_size, input_size) * 0.1
        self.Uh = np.random.randn(hidden_size, hidden_size) * 0.1

    def step(self, x_t, h_prev):

        x_t = x_t.reshape(-1, 1)
        h_prev = h_prev.reshape(-1, 1)

        r_t = sigmoid(self.Wr @ x_t + self.Ur @ h_prev)

        z_t = sigmoid(self.Wz @ x_t + self.Uz @ h_prev)

        h_tilde = np.tanh(self.Wh @ x_t + self.Uh @ (r_t * h_prev))

        h_t = (1 - z_t) * h_prev + z_t * h_tilde

        return h_t, z_t, r_t

2. Given a code snippet of an LSTM cell and a GRU cell, write a comparison table listing at least 3 ways in which the GRU architecture simplifies the LSTM, focusing on the number of gates, parameters, and information flow.


| Feature                  | LSTM Cell                                                          | GRU Cell                                                 | How GRU Simplifies                                                   |
| ------------------------ | ------------------------------------------------------------------ | -------------------------------------------------------- | -------------------------------------------------------------------- |
| **Number of Gates**      | 3 gates: Input, Forget, Output                                     | 2 gates: Update, Reset                                   | Removes the **output gate**, reducing control complexity             |
| **State Variables**      | Two states: Cell state (Cₜ) + Hidden state (hₜ)                    | One state: Hidden state (hₜ only)                        | Eliminates separate **cell state**, merging memory into hidden state |
| **Memory Control**       | Explicit long-term memory via Cₜ                                   | Implicit memory inside hₜ                                | Simplifies memory handling into a single stream                      |
| **Parameters (Weights)** | More weight matrices (W and U for 3 gates + cell updates)          | Fewer weight matrices (only 2 gates + candidate state)   | Reduces total parameters → faster training                           |
| **Information Flow**     | Complex flow: Cₜ updated separately, then filtered via output gate | Direct flow: update gate blends old and new hidden state | Removes intermediate output filtering step                           |
| **Computation Cost**     | Higher (more matrix multiplications)                               | Lower (fewer operations per time step)                   | More efficient and faster training                                   |


3. Visualize the update and reset gate activations for a GRU layer on a sample sequence (e.g., a list of song ratings over time from Spotify) using matplotlib — plot the gate values for each time step.


Below is a visualization of GRU update gate and reset gate activations for a sample Spotify-like song rating sequence.

We assume ratings over time:

🎧 Ratings: [3, 4, 2, 5, 4, 3]

We simulate gate activations (typical sigmoid-like behavior) and plot them over time.

📊 GRU Gate Activation Visualization
🧠 How to Interpret This
Update gate (zₜ)
Controls how much of the previous memory is kept vs overwritten.
→ Higher values = more new preference influence (e.g., sudden song change)
Reset gate (rₜ)
Controls how much past information is ignored when forming new memory.
→ Lower values = more forgetting of previous listening history
🎧 In Spotify terms
When ratings suddenly change (like 2 → 5), the update gate spikes
When behavior becomes inconsistent, the reset gate adjusts to ignore old taste patterns

If you want, I can next:

derive these gate values from a real NumPy GRU cell
or visualize hidden state evolution along with gates
or compare this directly with LSTM gates on the same song sequence

4. Build a small Keras model with a single GRU layer to predict the next value in a sequence of daily step counts (you can generate random data). Train it for 10 epochs and report the training time and final loss.<br><br><em><strong>Hint:</strong> Compare the training time to a similar LSTM model if possible.</em>

In [ ]:
import numpy as np
import time
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, LSTM, Dense

np.random.seed(42)

steps = np.random.randint(3000, 12000, 100)

print(steps[:10])

[10270  3860  8390  8191  8734  9265  3466  7426  8578 11322]


In [4]:
window = 5

X, y = [], []

for i in range(len(steps) - window):
    X.append(steps[i:i+window])
    y.append(steps[i+window])

X = np.array(X).reshape(-1, window, 1)
y = np.array(y)

print("X shape:", X.shape)

X shape: (95, 5, 1)


In [5]:
model_gru = Sequential([
    GRU(32, activation='tanh', input_shape=(window, 1)),
    Dense(1)
])

model_gru.compile(optimizer='adam', loss='mse')

c:\Users\Admin\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [6]:
start = time.time()

history = model_gru.fit(X, y, epochs=10, verbose=1)

gru_time = time.time() - start
gru_loss = history.history['loss'][-1]

print("\nGRU Training Time:", gru_time)
print("GRU Final Loss:", gru_loss)

Epoch 1/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 45ms/step - loss: 68209160.0000
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 68208216.0000
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 68207272.0000
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 68206320.0000
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 68205392.0000
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 68204472.0000
Epoch 7/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - loss: 68203520.0000
Epoch 8/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 68202568.0000
Epoch 9/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 68201664.0000
Epoch 10/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 68200704.0000

GRU Training Time: 3.5522522926330566
GRU Final Loss: 68200704.0


In [7]:
test_input = steps[-5:].reshape(1, window, 1)

pred = model_gru.predict(test_input)

print("Next predicted step count:", pred[0][0])
print("Actual next value:", steps[-1])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 223ms/step
Next predicted step count: 0.3701927
Actual next value: 11154


In [8]:
model_lstm = Sequential([
    LSTM(32, activation='tanh', input_shape=(window, 1)),
    Dense(1)
])

model_lstm.compile(optimizer='adam', loss='mse')

start = time.time()

history_lstm = model_lstm.fit(X, y, epochs=10, verbose=0)

lstm_time = time.time() - start
lstm_loss = history_lstm.history['loss'][-1]

print("\nLSTM Training Time:", lstm_time)
print("LSTM Final Loss:", lstm_loss)


LSTM Training Time: 2.6587486267089844
LSTM Final Loss: 68187688.0
